# Fraud Detection for Vehicle Insurance Claims
## Baseline-модели

In [9]:
# Импортируем библиотеки, подключаем корень проекта и функции из src.
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'data').exists():
            return candidate
    raise FileNotFoundError('Не удалось найти корень проекта (ожидались папки src и data).')


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.modeling import build_preprocessor, evaluate_model_dict
from src.preprocessing import prepare_datasets

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Загрузка processed-данных

In [10]:
# Загружаем processed-датасеты.
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'data').exists():
            return candidate
    raise FileNotFoundError('Не удалось найти корень проекта (ожидались папки src и data).')


PROJECT_ROOT = find_project_root(Path.cwd())
processed_dir = PROJECT_ROOT / 'data' / 'processed'
clean_path = processed_dir / 'fraud_oracle_clean.csv'
fe_path = processed_dir / 'fraud_oracle_fe.csv'

if not clean_path.exists() or not fe_path.exists():
    prepare_datasets(save=True)

clean_df = pd.read_csv(clean_path)
fe_df = pd.read_csv(fe_path)

target_col = 'FraudFound_P'

print('Загружено:', clean_path.as_posix())
print('Загружено:', fe_path.as_posix())
print('clean_df shape:', clean_df.shape)
print('fe_df shape:', fe_df.shape)

Загружено: c:/Users/Владислав/Desktop/ML/test/hseml-group-project-c0ldesty/data/processed/fraud_oracle_clean.csv
Загружено: c:/Users/Владислав/Desktop/ML/test/hseml-group-project-c0ldesty/data/processed/fraud_oracle_fe.csv
clean_df shape: (15419, 31)
fe_df shape: (15419, 37)


## 2. Train / Validation / Test split

In [11]:
# Готовим raw и fe выборки и делим на train/val/test.
X_raw = clean_df.drop(columns=[target_col])
y = clean_df[target_col]
X_fe = fe_df.drop(columns=[target_col])

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_train_raw, y_train, test_size=0.25, stratify=y_train, random_state=RANDOM_STATE
)

X_train_fe = X_fe.loc[X_train_raw.index]
X_val_fe = X_fe.loc[X_val_raw.index]
X_test_fe = X_fe.loc[X_test_raw.index]

print('RAW train:', X_train_raw.shape, y_train.shape)
print('RAW val:', X_val_raw.shape, y_val.shape)
print('RAW test:', X_test_raw.shape, y_test.shape)

RAW train: (9251, 30) (9251,)
RAW val: (3084, 30) (3084,)
RAW test: (3084, 30) (3084,)


## 3. Препроцессинг

In [12]:
# Собираем препроцессоры для raw и fe наборов признаков.
linear_preprocessor_raw, tree_preprocessor_raw, num_raw, cat_raw = build_preprocessor(X_train_raw)
linear_preprocessor_fe, tree_preprocessor_fe, num_fe, cat_fe = build_preprocessor(X_train_fe)

print('RAW: numeric =', len(num_raw), ', categorical =', len(cat_raw))
print('FE: numeric =', len(num_fe), ', categorical =', len(cat_fe))

RAW: numeric = 6 , categorical = 24
FE: numeric = 11 , categorical = 25


## 4. Метрики качества

Для этой задачи основной метрикой выбираем `PR-AUC`, потому что датасет несбалансирован: fraud-случаев заметно меньше, чем обычных заявок. В такой ситуации `accuracy` может быть высокой даже у модели, которая почти не находит положительный класс.

Дополнительно анализируем:

- `Recall` — показывает, какую долю fraud-случаев модель действительно находит. Для fraud detection это важная метрика, если в приоритет ставить поимку мошенничества.
- `F1` — помогает оценить баланс между найденными fraud-случаями и количеством ложных срабатываний.
- `Balanced Accuracy` — нужна как более честная альтернатива обычной accuracy на несбалансированном датасете, потому что учитывает качество по обоим классам.

Таким образом, в baseline-сравнении мы в первую очередь ориентируемся на `PR-AUC`, а `Recall`, `F1` и `Balanced Accuracy` используем для интерпретации того, как именно ведёт себя каждая модель.

## 5. Baseline

Строим baseline без engineered features.  
Используем простые модели:
- Dummy
- Logistic Regression
- LinearSVC
- KNN
- Decision Tree

In [13]:
# Определяем baseline-модели для raw-признаков.
baseline_models_raw = {
    'Dummy': Pipeline([
        ('preprocessor', linear_preprocessor_raw),
        ('model', DummyClassifier(strategy='stratified', random_state=RANDOM_STATE)),
    ]),
    'LogisticRegression': Pipeline([
        ('preprocessor', linear_preprocessor_raw),
        ('model', LogisticRegression(max_iter=3000, class_weight='balanced', random_state=RANDOM_STATE)),
    ]),
    'LinearSVC': Pipeline([
        ('preprocessor', linear_preprocessor_raw),
        ('model', LinearSVC(class_weight='balanced', random_state=RANDOM_STATE)),
    ]),
    'KNN': Pipeline([
        ('preprocessor', linear_preprocessor_raw),
        ('model', KNeighborsClassifier(n_neighbors=5, weights='distance')),
    ]),
    'DecisionTree': Pipeline([
        ('preprocessor', tree_preprocessor_raw),
        ('model', DecisionTreeClassifier(class_weight='balanced', random_state=RANDOM_STATE)),
    ]),
}

In [14]:
# Обучаем и оцениваем baseline-модели на raw-признаках.
baseline_raw_df, baseline_raw_fitted = evaluate_model_dict(
    baseline_models_raw,
    X_train_raw, y_train,
    X_val_raw, y_val,
    stage_name="baseline_raw",
    feature_set_name="raw",
)

display(baseline_raw_df.sort_values(by=["pr_auc", "recall", "f1"], ascending=False).style.format(precision=4))

,feature_set,stage,model,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,pr_auc
1,raw,baseline_raw,LogisticRegression,0.6625,0.7445,0.1329,0.8378,0.2295,0.8013,0.1582
2,raw,baseline_raw,LinearSVC,0.6495,0.7503,0.1316,0.8649,0.2284,0.7926,0.1579
3,raw,baseline_raw,KNN,0.9364,0.5183,0.2963,0.0432,0.0755,0.6169,0.1098
4,raw,baseline_raw,DecisionTree,0.8975,0.5786,0.1896,0.2162,0.2020,0.5786,0.0880
0,raw,baseline_raw,Dummy,0.8826,0.5024,0.0640,0.0703,0.0670,0.5024,0.0603


### Вывод по baseline на raw-признаках

- `Dummy` служит самой простой точкой отсчёта: качество у него низкое, PR-AUC и F1 минимальны, поэтому он показывает скорее базовый уровень задачи, чем полезный результат.
- `LogisticRegression` показывает один из лучших результатов среди baseline-моделей на raw-признаках: модель хорошо (по сравнению с остальными) находит fraud по recall, но делает это ценой низкого precision.
- `LinearSVC` ведёт себя похоже на логистическую регрессию: высокий recall, сопоставимый PR-AUC, но также много ложных срабатываний.
- `KNN` на raw-признаках работает слабо: несмотря на высокую accuracy, модель почти не выбирает fraud, поэтому recall и F1 остаются низкими.
- `DecisionTree` лучше улавливает fraud, чем Dummy, но всё ещё заметно уступает линейным моделям по PR-AUC, recall и устойчивости.

*Модели для первой проверки были взяты с минимальными, но не нулевыми настройками. Так без class_weight='balanced' все модели почти не выбирали fraud случаи, так что для примера я вывел их на хоть какой-то результат, значитильно отличный от Dummy.

Итог: на raw-признаках наиболее полезными baseline-моделями выглядят LogisticRegression и LinearSVC, потому что именно они лучше всего справляются с редким положительным классом. 

*Далее в cp2 после подбора дополнительных параметров DecisionTree показала лучший результат из моделей выше, так что сейчас мы получили результат с минимальными настройками, чтобы показать именно baseline модель.

## 6. Baseline с feature engineering

Теперь проверяем, улучшают ли engineered features те же простые модели.

In [15]:
# Определяем baseline-модели для набора с feature engineering.
baseline_models_fe = {
    'LogisticRegression': Pipeline([
        ('preprocessor', linear_preprocessor_fe),
        ('model', LogisticRegression(class_weight='balanced', random_state=RANDOM_STATE)),
    ]),
    'LinearSVC': Pipeline([
        ('preprocessor', linear_preprocessor_fe),
        ('model', LinearSVC(class_weight='balanced', random_state=RANDOM_STATE)),
    ]),
    'KNN': Pipeline([
        ('preprocessor', linear_preprocessor_fe),
        ('model', KNeighborsClassifier(n_neighbors=5, weights='distance')),
    ]),
    'DecisionTree': Pipeline([
        ('preprocessor', tree_preprocessor_fe),
        ('model', DecisionTreeClassifier(class_weight='balanced', random_state=RANDOM_STATE)),
    ]),
}

In [16]:
# Обучаем и оцениваем baseline-модели на fe-признаках.
baseline_fe_df, baseline_fe_fitted = evaluate_model_dict(
    baseline_models_fe,
    X_train_fe, y_train,
    X_val_fe, y_val,
    stage_name="baseline_fe",
    feature_set_name="fe",
)

display(baseline_fe_df.sort_values(by=["pr_auc", "recall", "f1"], ascending=False).style.format(precision=4))

,feature_set,stage,model,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,pr_auc
0,fe,baseline_fe,LogisticRegression,0.6741,0.7583,0.1391,0.8541,0.2392,0.8041,0.1624
1,fe,baseline_fe,LinearSVC,0.6595,0.7531,0.1344,0.8595,0.2325,0.7966,0.1619
2,fe,baseline_fe,KNN,0.9345,0.5072,0.1600,0.0216,0.0381,0.6089,0.0973
3,fe,baseline_fe,DecisionTree,0.8979,0.5687,0.1782,0.1946,0.1860,0.5687,0.0830


### Вывод по baseline с feature engineering

- `LogisticRegression` после добавления engineered-признаков остаётся одной из самых сильных baseline-моделей: у неё высокий recall, хороший balanced_accuracy и один из лучших PR-AUC в этом блоке.
- `LinearSVC` снова показывает высокий recall и остаётся сильным кандидатом, но по совокупности метрик обычно немного уступает логистической регрессии.
- `KNN` даже после feature engineering остаётся слабой моделью для fraud detection. Высокая accuracy здесь достигается в основном за счёт одного и того же класса, тогда как recall и F1 остаются низкими.
- `DecisionTree` с engineered-признаками не показывает заметного преимущества: она по-прежнему даёт умеренный recall, но уступает линейным моделям по качеству ранжирования и общему балансу метрик.

Итог: feature engineering немного помогает baseline-моделям, но основная картина сохраняется. Лучшие результаты в этом блоке и с данными настройками дают `LogisticRegression` и `LinearSVC`, а `KNN` и `DecisionTree` остаются существенно слабее.

В телеграме писали, что в cp1 нужна только baseline модель, но я на всякий случай вынес сюда еще первый feature engineering. Более мощные модели и ансамбли, подбор гипперпараметров, эксперименты с размерностью, подбор threshold и финальное обучение лучшей модели я отнес в cp2. 